# DPO: concise professional email rewriting

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_preference_email.py`](../examples/run_preference_email.py).

Where our other DPO demo tunes support-reply *warmth*, this teaches **concise +
professional** email rewriting and records an **honest before → after** lift with
two disciplines borrowed from the RLFT demo:

1. A **pilot gate** — judge the untuned base rewrite against the *gold preferred*
   reference; if the base already matches the concise gold there is no headroom.
2. A **head-to-head** win-rate — the tuned rewrite vs the *base* rewrite of the
   same draft (not a fixed strawman), judged blind with a randomized A/B position,
   with a `bootstrap_ci`. `win_rate > 0.5` means tuning helped.

An objective compression ratio backs up the win-rate.

> **Requires live GCP and incurs tuning cost** (one preference-tuning job). Have
> a real `.env` and `gcloud auth` in place.

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"
JUDGE_MODEL = "gemini-2.5-flash"
SAT_CEILING = 0.6  # base win_rate vs gold must be below this to have headroom
cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the preference dataset and stage it to GCS

Each record is a `(draft, preferred, dispreferred)` triple — hand-authored so the
preferred rewrite is professional **and** materially shorter than the dispreferred
one (a concision invariant the unit tests enforce). The bank is 60 triples so the
0.25 test split gives ~15 held-out drafts — enough for a meaningful win-rate CI.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.preference.email import (
    EMAIL_DRAFTS,
    SYSTEM_INSTRUCTION,
    build_preference_dataset,
    build_preference_records,
    split_dataset,
)

GCS_PREFIX = "preference_concise_email_v2"
paths = build_preference_dataset("../datasets/preference_concise_email")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/{GCS_PREFIX}/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/{GCS_PREFIX}/val.jsonl")

_, _, test_triples = split_dataset(EMAIL_DRAFTS)
test_records = build_preference_records(test_triples)
print(f"{len(EMAIL_DRAFTS)} triples, {len(test_records)} held out")

## 2. The blind A/B judge and the base generator

The judge prompt is **different** from the generator's `SYSTEM_INSTRUCTION` and
reads only the first letter of the verdict. The eval helpers randomize which slot
(A or B) each candidate lands in, so the judge's position bias cancels out.

In [ ]:
from geap_tuning.inference import generate

_JUDGE_PROMPT = (
    "You are judging two versions of the same work email. Pick the version that "
    "is more professional and concise (clear, brief, free of filler and hedging) "
    "while keeping the same information. Answer with only the single letter 'A' "
    "or 'B'.\n\n"
    "Original draft: {user}\n\nEmail A: {a}\n\nEmail B: {b}\n\nBetter email:"
)


def judge_fn(draft: str, cand_a: str, cand_b: str) -> str:
    verdict = generate(client, JUDGE_MODEL, _JUDGE_PROMPT.format(user=draft, a=cand_a, b=cand_b))
    return verdict[:1].upper()


def base_rewrite(draft: str) -> str:
    return generate(client, BASE_MODEL, draft, system_instruction=SYSTEM_INSTRUCTION)

## 3. Pilot gate — base rewrite vs the gold preferred (the "before")

`run_pilot_eval` judges the base's own rewrite against the concise human gold. A
**low** win-rate means the base trails the gold (real headroom to tune); a high
one means it already writes as well as the gold (skip tuning).

In [ ]:
from geap_tuning.preference.email_eval import run_pilot_eval

pilot = run_pilot_eval(test_records, base_rewrite, judge_fn)
print(
    f"BASE vs gold: win_rate={pilot['win_rate']:.3f} (ceiling {SAT_CEILING}) "
    f"mean_compression={pilot['mean_compression']:.2f} (n={pilot['n']})"
)
if pilot["win_rate"] < SAT_CEILING:
    print("Pilot gate PASSED — the base trails the concise gold; headroom confirmed.")
else:
    print("WARNING: base already matches the gold (no headroom) — tuning may not show a lift.")

## 4. Launch the preference-tuning job and wait

A fresh display name (`-v2`) and a firmer pull toward the preferred (shorter)
completion than the defaults: `epochs=3`, `beta=0.2`.

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_endpoint, wait_for_tuning_job
from geap_tuning.preference.tune import launch_preference_job

DISPLAY_NAME = "geap-dpo-concise-email-v2"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_preference_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        epochs=3,
        beta=0.2,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)


def tuned_rewrite(draft: str) -> str:
    return generate(client, endpoint, draft, system_instruction=SYSTEM_INSTRUCTION)


endpoint

## 5. Head-to-head before → after and report the lift

`run_head_to_head_eval` judges the tuned rewrite against the **base** rewrite of
the same draft (blind, position-randomized). `win_rate > 0.5` with a `bootstrap_ci`
that clears 0.5 is a real lift; `run_pilot_eval` on the tuned model shows whether
it closed the gap to the gold.

In [ ]:
from geap_tuning.preference.email_eval import run_head_to_head_eval
from geap_tuning.rlft.evaluate import bootstrap_ci

h2h = run_head_to_head_eval(test_records, base_rewrite, tuned_rewrite, judge_fn)
tuned_pilot = run_pilot_eval(test_records, tuned_rewrite, judge_fn)
low, high = bootstrap_ci(int(h2h["hits"]), int(h2h["n"]))
print(
    f"HEAD-TO-HEAD tuned vs base: win_rate={h2h['win_rate']:.3f} "
    f"CI[{low:.3f}, {high:.3f}] (n={h2h['n']}); >0.5 means tuning helped"
)
print(
    f"compression base={h2h['base_mean_compression']:.2f} tuned={h2h['tuned_mean_compression']:.2f}"
)
print(f"vs gold: base={pilot['win_rate']:.3f} -> tuned={tuned_pilot['win_rate']:.3f}")